# Descarga de datos: Kalshi y BRTI

Descargo los contratos BTC de 15 minutos y las series necesarias para construir el dataset. Kalshi separa mercados recientes y archivados mediante un cutoff que puede cambiar con el tiempo. Por eso consulto ambos catálogos y selecciono el endpoint de velas según la liquidación de cada contrato.

El período se define en `START` y `END` (UTC, fin exclusivo). Los archivos ya descargados se reutilizan; al terminar reviso cuántos contratos se encontraron por mes y cuántas descargas fallaron.


In [1]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import local

import pandas as pd
import requests
from tqdm.auto import tqdm

DATA = Path('data/raw')
DATA.mkdir(parents=True, exist_ok=True)

# Período de estudio (UTC, fin exclusivo).
START = '2025-12-01T00:00:00+00:00'
END = '2026-11-01T00:00:00+00:00'
KALSHI_SERIES = 'KXBTC15M'
MAX_MARKETS = None  # None: no recortar el catálogo; definir un entero solo para pruebas
MAX_PAGES = 1000  # Si se alcanza, la descarga se detiene con error; nunca se acepta un catálogo parcial
DOWNLOAD_KALSHI_CANDLES = True
DOWNLOAD_KALSHI_TRADES = False  # El dataset original usaba velas, no trades individuales

# Concurrencia moderada para no saturar la API.
KALSHI_WORKERS = 2
REQUEST_RETRIES = 5
REQUEST_TIMEOUT = 45

start_ts = int(pd.Timestamp(START).timestamp())
end_ts = int(pd.Timestamp(END).timestamp())
assert end_ts > start_ts
print('Período:', START, '→', END)


Período: 2025-12-01T00:00:00+00:00 → 2026-11-01T00:00:00+00:00


/home/fernando/projects/udesa/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Funciones de descarga

Guardo el JSON original de cada respuesta. Así no pierdo campos cuando cambie el esquema o quiera hacer otro análisis.

In [2]:
import random
# Cada hilo utiliza su propia sesión HTTP. Evita compartir una Session entre hilos.
_thread = local()

def http_session():
    if not hasattr(_thread, 'session'):
        s = requests.Session()
        s.headers.update({'User-Agent': 'btc15m-academic-research/1.0'})
        _thread.session = s
    return _thread.session

def get_json(url, params=None, retries=REQUEST_RETRIES):
    for attempt in range(retries):
        try:
            r = http_session().get(url, params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code in (429, 500, 502, 503, 504):
                if attempt == retries - 1:
                    r.raise_for_status()
                retry_after = r.headers.get('Retry-After')
                try:
                    wait = float(retry_after) if retry_after else min(2 ** attempt, 30)
                except ValueError:
                    wait = min(2 ** attempt, 30)
                time.sleep(max(0, min(wait, 60)) + random.uniform(0, 0.5))
                continue
            r.raise_for_status()
            return r.json()
        except (requests.Timeout, requests.ConnectionError):
            if attempt == retries - 1:
                raise
            time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f'No se pudo descargar {url}')

def save_json(obj, path):
    path = DATA / path
    path.parent.mkdir(parents=True, exist_ok=True)
    # Reemplazo atómico: un corte no deja archivos JSON a medio escribir.
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)
    return path

def read_cached(path, check=None):
    path = DATA / path
    if not path.is_file():
        return None
    try:
        obj = json.loads(path.read_text(encoding='utf-8'))
        if check is not None and not check(obj):
            return None
        return obj
    except (OSError, ValueError, TypeError):
        return None

def paginated(url, params=None, field='markets', max_pages=MAX_PAGES):
    records, cursor = [], None
    for page in range(max_pages):
        q = dict(params or {})
        if cursor:
            q['cursor'] = cursor
        payload = get_json(url, q)
        batch = payload.get(field, [])
        if not isinstance(batch, list):
            raise ValueError(f'Respuesta inesperada en {url}: {field} no es lista')
        records.extend(batch)
        cursor = payload.get('cursor')
        if not cursor or not batch:
            return records, True
    return records, False

KALSHI = 'https://api.elections.kalshi.com/trade-api/v2'
GAMMA = 'https://gamma-api.polymarket.com'
CLOB = 'https://clob.polymarket.com'


## 1. Kalshi — descubrir contratos recientes e históricos

Consulto `/historical/cutoff` en cada ejecución. El catálogo histórico se pagina por `series_ticker`; como no admite filtros de fecha de cierre, filtro localmente. No supongo que el orden devuelto por la API sea cronológico ni recorto silenciosamente los mercados encontrados.


In [3]:
cutoff = get_json(f'{KALSHI}/historical/cutoff')
save_json(cutoff, 'kalshi/cutoff.json')
print(cutoff)

live, live_complete = paginated(
    f'{KALSHI}/markets',
    {'series_ticker': KALSHI_SERIES, 'limit': 1000, 'mve_filter': 'exclude',
     'min_close_ts': start_ts, 'max_close_ts': end_ts},
    max_pages=MAX_PAGES
)

# Los archivados se listan por serie; la API no garantiza orden temporal.
archived, archived_complete = paginated(
    f'{KALSHI}/historical/markets',
    {'series_ticker': KALSHI_SERIES, 'limit': 1000},
    max_pages=MAX_PAGES
)

def close_unix(m):
    x = m.get('close_time') or m.get('expected_expiration_time')
    if not x:
        return None
    try:
        return int(pd.Timestamp(x).timestamp())
    except (ValueError, TypeError):
        return None

if not (live_complete and archived_complete):
    raise RuntimeError('Paginación Kalshi incompleta. No sobrescribo markets.json con un catálogo parcial.')

# Identifico explícitamente de qué catálogo proviene cada mercado.
# En caso de duplicados, el registro archivado prevalece para los ya archivados.
markets_by_ticker = {}
for source, records in [('live', live), ('historical', archived)]:
    for m in records:
        ticker = m.get('ticker')
        close = close_unix(m)
        if ticker and close is not None and start_ts <= close < end_ts:
            markets_by_ticker[ticker] = {**m, '_catalog_source': source}

markets = sorted(markets_by_ticker.values(), key=lambda m: (close_unix(m), m['ticker']))
found_count = len(markets)
if MAX_MARKETS is not None:
    if not isinstance(MAX_MARKETS, int) or MAX_MARKETS <= 0:
        raise ValueError('MAX_MARKETS debe ser None o un entero positivo')
    if found_count > MAX_MARKETS:
        raise RuntimeError(f'Se encontraron {found_count} mercados, pero MAX_MARKETS={MAX_MARKETS}. '
                           'Para una descarga completa usá MAX_MARKETS=None.')

historical_count = sum(m['_catalog_source'] == 'historical' for m in markets)
print('Mercados recientes:', len(live), '| archivados:', len(archived))
print('Mercados del período:', found_count, '| encontrados en archivo histórico:', historical_count)
if markets:
    months = pd.Series([pd.Timestamp(close_unix(m), unit='s', tz='UTC').strftime('%Y-%m')
                        for m in markets]).value_counts().sort_index()
    print('Contratos por mes (fecha de cierre UTC):')
    print(months.to_string())
    print('Primero:', markets[0]['ticker'], '| último:', markets[-1]['ticker'])

save_json({'markets': markets, 'query': {'start': START, 'end': END,
          'series': KALSHI_SERIES}, 'live_pagination_complete': live_complete,
          'archived_pagination_complete': archived_complete,
          'market_count': found_count}, 'kalshi/markets.json')
print('Mercados:', len(markets))
print('Paginación completa:', live_complete, archived_complete)
if not markets:
    print('No hubo resultados: revisar KALSHI_SERIES, fechas y límites de paginación.')

{'market_positions_last_updated_ts': '2026-09-17T00:00:00Z', 'market_settled_ts': '2026-09-17T00:00:00Z', 'orders_updated_ts': '2026-11-02T00:00:00Z', 'trades_created_ts': '2026-09-17T00:00:00Z'}
Mercados recientes: 5217 | archivados: 26117
Mercados del período: 30381 | encontrados en archivo histórico: 26117
Contratos por mes (fecha de cierre UTC):
2025-12    1566
2026-01    2972
2026-02    2676
2026-03    2928
2026-04    2820
2026-05    2928
2026-06    2835
2026-07    2931
2026-08    2941
2026-09    2848
2026-10    2936
Primero: KXBTC15M-25DEC101700-00 | último: KXBTC15M-26OCT311945-45
Mercados: 30381
Paginación completa: True True


## 2. Kalshi — velas (paralelo y reanudable)

Cada contrato se procesa en un hilo, con un máximo de `KALSHI_WORKERS` peticiones simultáneas. Los JSON ya descargados se reutilizan al volver a ejecutar esta celda. Solo se considera completo un archivo de trades cuando terminó toda su paginación. Si una petición falla, queda registrada para reintentar en la siguiente ejecución.

Los archivos se guardan con los mismos nombres que antes, dentro de `data/raw/kalshi/`.


In [4]:
market_by_ticker = {m['ticker']: m for m in markets if m.get('ticker')}

def cutoff_seconds(value):
    if isinstance(value, (int, float)):
        return int(value)
    return int(pd.Timestamp(value).timestamp()) if value else None

market_cutoff = cutoff_seconds(cutoff.get('market_settled_ts'))
trade_cutoff = cutoff_seconds(cutoff.get('trades_created_ts'))

def process_kalshi_market(market):
    ticker = market['ticker']
    close = close_unix(market)
    window_start = max(start_ts, close - 15 * 60)
    window_end = min(end_ts, close + 60)
    row = {'ticker': ticker, 'catalog_source': market_by_ticker.get(ticker, {}).get('_catalog_source'), 'trades': 0, 'candles': 0,
           'cached_trades': 0, 'cached_candles': False, 'errors': []}

    if DOWNLOAD_KALSHI_TRADES:
        segments = []
        if trade_cutoff and window_start < trade_cutoff:
            segments.append(('historical', window_start, min(window_end, trade_cutoff)))
        if trade_cutoff is None or window_end > trade_cutoff:
            segments.append(('live', max(window_start, trade_cutoff or window_start), window_end))
        for kind, a, z in segments:
            if z <= a:
                continue
            relpath = f'kalshi/trades/{ticker}_{kind}.json'
            def valid_trades(x):
                return (isinstance(x, dict) and x.get('ticker') == ticker and
                        x.get('source') == kind and x.get('pagination_complete') is True and
                        isinstance(x.get('trades'), list) and
                        (x.get('window_start') in (None, a)) and
                        (x.get('window_end') in (None, z)))
            cached = read_cached(relpath, valid_trades)
            if cached is not None:
                row['trades'] += len(cached['trades'])
                row['cached_trades'] += 1
                continue
            endpoint = '/historical/trades' if kind == 'historical' else '/markets/trades'
            try:
                trades, complete = paginated(f'{KALSHI}{endpoint}',
                    {'ticker': ticker, 'min_ts': a, 'max_ts': z, 'limit': 1000}, field='trades')
                if not complete:
                    raise RuntimeError(f'{kind} trades: paginación incompleta; no se guarda como válido')
                save_json({'ticker': ticker, 'source': kind, 'trades': trades,
                           'pagination_complete': True, 'window_start': a, 'window_end': z}, relpath)
                row['trades'] += len(trades)
            except (requests.RequestException, RuntimeError, ValueError, TypeError) as e:
                row['errors'].append(f'{kind} trades: {e}')

    if DOWNLOAD_KALSHI_CANDLES:
        relpath = f'kalshi/candles/{ticker}.json'
        def valid_candles(x):
            return isinstance(x, dict) and isinstance(x.get('candlesticks'), list)
        cached = read_cached(relpath, valid_candles)
        if cached is not None:
            row['candles'] = len(cached['candlesticks'])
            row['cached_candles'] = True
            row['candles_source'] = 'cached'
        else:
            # El cutoff corresponde a la liquidación, no necesariamente al cierre.
            settlement = market.get('settlement_ts')
            if settlement:
                try:
                    settlement_ts = int(pd.Timestamp(settlement).timestamp())
                except (TypeError, ValueError):
                    settlement_ts = None
            else:
                settlement_ts = None
            is_archived = (market.get('_catalog_source') == 'historical' or
                           (market_cutoff is not None and settlement_ts is not None
                            and settlement_ts < market_cutoff))
            endpoint = (f'/historical/markets/{ticker}/candlesticks' if is_archived else
                        f'/series/{KALSHI_SERIES}/markets/{ticker}/candlesticks')
            try:
                payload = get_json(f'{KALSHI}{endpoint}',
                    {'start_ts': window_start, 'end_ts': window_end, 'period_interval': 1})
                if not valid_candles(payload):
                    raise ValueError('Respuesta sin candlesticks válidos')
                save_json(payload, relpath)
                row['candles'] = len(payload['candlesticks'])
                row['candles_source'] = 'historical' if is_archived else 'live'
            except (requests.RequestException, ValueError, TypeError) as e:
                row['errors'].append(f'candles: {e}')
    return row

# as_completed permite mostrar el avance a medida que termina cada contrato.
manifest = []
with ThreadPoolExecutor(max_workers=KALSHI_WORKERS) as pool:
    futures = {pool.submit(process_kalshi_market, m): m['ticker'] for m in markets}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Kalshi'):
        ticker = futures[future]
        try:
            row = future.result()
        except Exception as e:
            row = {'ticker': ticker, 'catalog_source': market_by_ticker.get(ticker, {}).get('_catalog_source'), 'trades': 0, 'candles': 0,
                   'cached_trades': 0, 'cached_candles': False,
                   'errors': [f'Error inesperado: {type(e).__name__}: {e}']}
        manifest.append(row)

manifest.sort(key=lambda x: x['ticker'])
save_json(manifest, 'kalshi/download_manifest.json')
errors = [r for r in manifest if r['errors']]
print('Contratos:', len(manifest), '| con errores:', len(errors))
print('Velas históricas nuevas:', sum(r.get('candles_source') == 'historical' for r in manifest))
print('Contratos sin velas:', sum(r['candles'] == 0 for r in manifest))
print('Trades en caché:', sum(r['cached_trades'] for r in manifest),
      '| velas en caché:', sum(r['cached_candles'] for r in manifest))
if errors:
    print('Primeros errores:', errors[:5])
pd.DataFrame(manifest).head()


Kalshi: 100%|███████████████████████████████████████████████████████████████| 30381/30381 [00:02<00:00, 12115.81it/s]


Contratos: 30381 | con errores: 0
Velas históricas nuevas: 0
Contratos sin velas: 104
Trades en caché: 0 | velas en caché: 30381


,ticker,catalog_source,trades,candles,cached_trades,cached_candles,errors,candles_source
0,KXBTC15M-25DEC101700-00,historical,0,0,0,True,[],cached
1,KXBTC15M-25DEC101715-15,historical,0,2,0,True,[],cached
2,KXBTC15M-25DEC101730-30,historical,0,0,0,True,[],cached
3,KXBTC15M-25DEC101745-45,historical,0,0,0,True,[],cached
4,KXBTC15M-25DEC101800-00,historical,0,0,0,True,[],cached


**Control de cobertura.** Los mercados archivados se buscan mediante el catálogo histórico, y el resumen por mes permite comprobar si realmente se incorporó el período anterior al cutoff. Una descarga sin errores HTTP no garantiza cobertura completa: también reviso contratos sin velas y posibles huecos temporales en el Notebook 02. Los archivos anteriores no se borran.

## 4. BRTI — ticks originales desde Kalshi

Uso el endpoint histórico de CF Benchmarks disponible a través de Kalshi, como en el script original `descargar_brti_30d.py`:

`GET https://external-api.kalshi.com/trade-api/v2/cfbenchmarks/history/values`

La petición requiere **API key ID** y firma de cada solicitud con la **clave privada** (RSA-PSS o Ed25519). Las credenciales se leen de variables de entorno; no se guardan en el notebook.

Se descarga **una hora por petición**, con el mismo período de 30 días definido arriba (720 horas). Con una frecuencia de 5 Hz esperamos 18.000 ticks por hora, aproximadamente 12.960.000 en total. El código guarda cada hora en un JSON comprimido, reanuda las descargas y produce un reporte. Los valores de precio se conservan como cadenas para no introducir redondeos.

Para una primera prueba, usar `BRTI_MAX_HOURS = 2`; para los 30 días completos, `0`.


In [5]:
import base64
import gzip
import random
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ed25519, padding

BRTI_BASE = 'https://external-api.kalshi.com'
BRTI_PATH = '/trade-api/v2/cfbenchmarks/history/values'
BRTI_MAX_HOURS = 0  # 0 = las 720 horas; para prueba usar 2 explícitamente
BRTI_RETRIES = 5
BRTI_SLEEP = 0.35
BRTI_RETRY_INCOMPLETE = False  # Reconsultar horas incompletas, conservando originales si no mejora

key_id = os.getenv('KALSHI_API_KEY_ID')
private_key_path = os.getenv('KALSHI_PRIVATE_KEY_PATH')


def sign_brti(key, key_id):
    stamp = str(int(time.time() * 1000))
    message = f'{stamp}GET{BRTI_PATH}'.encode()
    if isinstance(key, ed25519.Ed25519PrivateKey):
        signature = key.sign(message)
    else:
        signature = key.sign(
            message,
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
            hashes.SHA256(),
        )
    return {
        'KALSHI-ACCESS-KEY': key_id,
        'KALSHI-ACCESS-TIMESTAMP': stamp,
        'KALSHI-ACCESS-SIGNATURE': base64.b64encode(signature).decode(),
    }


def read_brti_hour(path, start_ms):
    try:
        with gzip.open(path, 'rt', encoding='utf-8') as f:
            data = json.load(f)
        if data.get('hour_start_ms') != start_ms or not data.get('values'):
            return None
        return data
    except (OSError, ValueError, KeyError, TypeError):
        return None


def download_brti(max_hours=0):
    if not key_id or not private_key_path:
        raise RuntimeError('Definí KALSHI_API_KEY_ID y KALSHI_PRIVATE_KEY_PATH antes de ejecutar esta celda.')
    key = serialization.load_pem_private_key(Path(private_key_path).expanduser().read_bytes(), password=None)
    start = datetime.fromisoformat(START)
    end = datetime.fromisoformat(END)
    assert start.tzinfo is not None and end.tzinfo is not None
    assert all((start.minute == 0, start.second == 0, end.minute == 0, end.second == 0))
    hours = int((end - start).total_seconds() // 3600)
    if max_hours:
        hours = min(hours, max_hours)
    out = DATA / 'brti'
    out.mkdir(parents=True, exist_ok=True)
    failed, incomplete = [], []
    retried_incomplete, recovered_ticks, unchanged_incomplete = [], 0, []
    total = 0
    for i in tqdm(range(hours), desc='BRTI por hora'):
        hour = start + timedelta(hours=i)
        hour_ms = int(hour.timestamp() * 1000)
        dest = out / (hour.strftime('%Y%m%dT%HZ') + '.json.gz')
        cached = read_brti_hour(dest, hour_ms) if dest.exists() else None
        cached_values = cached['values'] if cached is not None else []
        cached_times = {int(v[0]) for v in cached_values}
        cached_complete = (len(cached_values) == 18000 and len(cached_times) == 18000)
        if cached_complete or (cached is not None and not BRTI_RETRY_INCOMPLETE):
            total += len(cached_values)
            if not cached_complete:
                incomplete.append(dest.name)
            continue
        if cached is not None:
            retried_incomplete.append(dest.name)
        params = {'id': 'BRTI', 'timespan': 'HOUR', 'timestamp': hour.strftime('%Y-%m-%dT%H:%M:%S.000Z')}
        for attempt in range(BRTI_RETRIES + 1):
            try:
                response = requests.get(
                    BRTI_BASE + BRTI_PATH,
                    params=params,
                    headers=sign_brti(key, key_id),
                    timeout=70,
                )
                response.raise_for_status()
                body = response.json()
                payload = body.get('data', {}).get('payload')
                if not isinstance(payload, list) or not payload:
                    raise ValueError(f'Payload BRTI vacío o inesperado: {str(body)[:250]}')
                values = [[int(x['time']), str(x['value'])] for x in payload]
                if any(not hour_ms <= t < hour_ms + 3600000 for t, _ in values):
                    raise ValueError('Hay ticks fuera de la hora solicitada')
                values.sort(key=lambda x: x[0])
                unique = len({v[0] for v in values})
                data = {
                    'index': 'BRTI', 'hour_start_ms': hour_ms,
                    'source': 'Kalshi CF Benchmarks historical values',
                    'values': values, 'count': len(values),
                    'unique_timestamps': unique,
                    'first_time_ms': values[0][0], 'last_time_ms': values[-1][0],
                }
                # Si existe una respuesta anterior, conservar la unión de ticks.
                # Nunca reemplazar una hora por una respuesta con menor cobertura.
                if cached is not None:
                    merged = {int(t): str(p) for t, p in cached_values}
                    previous_count = len(merged)
                    for t, p in values:
                        merged.setdefault(int(t), str(p))
                    recovered_ticks += len(merged) - previous_count
                    values = [[t, merged[t]] for t in sorted(merged)]
                    unique = len(merged)
                    data['values'] = values
                    data['count'] = len(values)
                    data['unique_timestamps'] = unique
                    data['first_time_ms'] = values[0][0]
                    data['last_time_ms'] = values[-1][0]
                    if len(merged) == previous_count:
                        unchanged_incomplete.append(dest.name)
                tmp = out / (dest.name + '.tmp')
                with gzip.open(tmp, 'wt', encoding='utf-8', compresslevel=6) as f:
                    json.dump(data, f, separators=(',', ':'))
                tmp.replace(dest)
                total += len(values)
                if len(values) != 18000 or unique != 18000:
                    incomplete.append(dest.name)
                break
            except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
                if attempt == BRTI_RETRIES:
                    failed.append({'hour': hour.isoformat(), 'error': str(exc)})
                    if cached is not None:
                        total += len(cached_values)
                        incomplete.append(dest.name)
                    print('Falló:', hour.isoformat(), str(exc)[:250])
                else:
                    time.sleep(min(60, 2 ** attempt + random.random()))
        time.sleep(BRTI_SLEEP)
    report = {
        'start': START, 'end_requested': END,
        'hours_requested': hours,
        'ticks_total_including_cached': total,
        'expected_ticks_at_5hz': hours * 18000,
        'failed_hours': failed,
        'incomplete_hours': sorted(set(incomplete)),
        'retried_incomplete_hours': retried_incomplete,
        'recovered_timestamps': recovered_ticks,
        'unchanged_incomplete_hours': unchanged_incomplete,
        'note': 'La existencia de ticks históricos no demuestra latencia de publicación en vivo.',
    }
    (out / 'download_report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print('Reporte:', out / 'download_report.json')
    print('Ticks:', total, '/', hours * 18000, '| horas fallidas:', len(failed))
    print('Horas incompletas reconsultadas:', len(retried_incomplete),
          '| timestamps recuperados:', recovered_ticks,
          '| horas sin nuevos ticks:', len(unchanged_incomplete))
    return report

# Variables de entorno, por ejemplo, antes de iniciar Jupyter:
# export KALSHI_API_KEY_ID='tu_key_id'
# export KALSHI_PRIVATE_KEY_PATH='/ruta/a/tu_clave_privada.pem'

if key_id and private_key_path:
    brti_report = download_brti(max_hours=BRTI_MAX_HOURS)
else:
    print('BRTI sin ejecutar: configurá KALSHI_API_KEY_ID y KALSHI_PRIVATE_KEY_PATH.')


BRTI por hora: 100%|█████████████████████████████████████████████████████████████| 8040/8040 [02:07<00:00, 63.21it/s]

Reporte: data/raw/brti/download_report.json
Ticks: 86325193 / 144720000 | horas fallidas: 0
Horas incompletas reconsultadas: 0 | timestamps recuperados: 0 | horas sin nuevos ticks: 0


## 5. Control de descarga

Este control no calcula features: solo enumera los archivos disponibles y deja documentado qué fuentes faltan. Si la paginación se corta por `MAX_PAGES`, los datos pueden estar incompletos.

In [6]:
files = sorted(p for p in DATA.rglob('*') if p.is_file())
summary = pd.DataFrame([{'archivo': str(p), 'bytes': p.stat().st_size} for p in files])
display(summary)
print('Total de archivos:', len(files))
print('Kalshi mercados:', len(markets))
print('BRTI disponible:', bool(list((DATA / 'brti').glob('*'))) if (DATA / 'brti').exists() else False)

,archivo,bytes
0,data/raw/brti/20251201T00Z.json.gz,22275
1,data/raw/brti/20251201T01Z.json.gz,21611
2,data/raw/brti/20251201T02Z.json.gz,21324
3,data/raw/brti/20251201T03Z.json.gz,20767
4,data/raw/brti/20251201T04Z.json.gz,20854
...,...,...
163894,data/raw/polymarket/settlements/btc-updown-15m...,817
163895,data/raw/polymarket/settlements/btc-updown-15m...,817
163896,data/raw/polymarket/settlements/btc-updown-15m...,816
163897,data/raw/polymarket/settlements/btc-updown-15m...,817


Total de archivos: 163899
Kalshi mercados: 30381
BRTI disponible: True


In [7]:
# Auditoría de cobertura: un archivo presente no implica datos útiles.
from collections import Counter
from pathlib import Path
import gzip

expected_hours = int((pd.Timestamp(END)-pd.Timestamp(START)).total_seconds()//3600)
files_brti = sorted((DATA/'brti').glob('*.json.gz'))
valid_hours, bad_hours = 0, []
for p in files_brti:
    try:
        with gzip.open(p,'rt',encoding='utf-8') as f:
            h=json.load(f)
        vals=h.get('values',[])
        if len(vals)==18000 and len({x[0] for x in vals})==18000:
            valid_hours += 1
        else:
            bad_hours.append((p.name,len(vals)))
    except (OSError, ValueError, TypeError) as exc:
        bad_hours.append((p.name,str(exc)))

km = read_cached('kalshi/download_manifest.json') or []
pm = read_cached('polymarket/download_manifest.json') or []
print('BRTI horas completas:',valid_hours,'/',expected_hours,'| incompletas:',len(bad_hours))
print('Kalshi mercados con velas:',sum(x.get('candles',0)>0 for x in km),'/',len(km))
print('Kalshi errores:',sum(bool(x.get('errors')) for x in km))
print('Polymarket ventanas con eventos:',sum(x.get('events',0)>0 for x in pm),'/',len(pm))
print('Polymarket puntos de historial reportados:',sum(x.get('points',0) for x in pm))
print('Polymarket ventanas con errores:',sum(bool(x.get('errors')) for x in pm))
if valid_hours < expected_hours:
    print('ADVERTENCIA: faltan horas BRTI completas. No considerar completa la muestra de 30 días.')
if bad_hours:
    print('Primeras horas incompletas:',bad_hours[:5])


BRTI horas completas: 3794 / 8040 | incompletas: 4246
Kalshi mercados con velas: 30277 / 30381
Kalshi errores: 0
Polymarket ventanas con eventos: 32031 / 32064
Polymarket puntos de historial reportados: 956898
Polymarket ventanas con errores: 0
ADVERTENCIA: faltan horas BRTI completas. No considerar completa la muestra de 30 días.
Primeras horas incompletas: [('20251201T00Z.json.gz', 3600), ('20251201T01Z.json.gz', 3600), ('20251201T02Z.json.gz', 3600), ('20251201T03Z.json.gz', 3600), ('20251201T04Z.json.gz', 3600)]


## Siguiente paso

En el notebook 02 construyo las observaciones por contrato y minuto, alineo los precios con BRTI y Kalshi y reviso la calidad de los datos. Antes de usarlos, verifico que las descargas cubran el período elegido y que no haya páginas pendientes.
